# `GoapSubgraph` quickstart

Drop a GOAP planning loop into an existing LangGraph application as a single sealed node. This primer covers **Layer C**: hand-author a few `ActionSpec` objects, wrap them in a `GoapSubgraph`, and either run it standalone or attach it to a parent `StateGraph` via `add_goap_subgraph`. The parent only sees the keys you pick (defaults: `world_state` in, `plan_result` out); GOAP-private state (`plan`, `current_step`, `execution_history`, ...) stays inside the sub-graph.

The cells below mirror [`tests/integration/test_subgraph.py::TestSubgraphQuickstart`](../../tests/integration/test_subgraph.py) exactly, so this notebook is regression-tested on every CI run.

## 1. Three hand-authored actions

A minimal pipeline: `research → write → publish`. Each `execute` callable returns *both* the data the next step consumes (`brief`, `draft`, `url`) and the boolean flag that satisfies its declared effect (`have_brief`, `have_draft`, `published`) — those flags are what A\* searches over.

In [1]:
from langgoap import ActionSpec, GoalSpec


def quickstart_actions():
    return [
        ActionSpec(
            name="research",
            preconditions={},
            effects={"have_brief": True},
            execute=lambda ws: {
                "brief": f"Brief on {ws['topic']}",
                "have_brief": True,
            },
            cost=1.0,
        ),
        ActionSpec(
            name="write",
            preconditions={"have_brief": True},
            effects={"have_draft": True},
            execute=lambda ws: {
                "draft": f"Article from: {ws['brief']}",
                "have_draft": True,
            },
            cost=1.0,
        ),
        ActionSpec(
            name="publish",
            preconditions={"have_draft": True},
            effects={"published": True},
            execute=lambda ws: {
                "url": f"https://blog/{ws['draft'][:10]}",
                "published": True,
            },
            cost=1.0,
        ),
    ]


goal = GoalSpec(conditions={"published": True})

## 2. Run the sub-graph standalone

`GoapSubgraph(...).compile()` returns a `CompiledStateGraph` whose schema is the GOAP-internal `GoapState`. You pass `world_state` + `goal` in and read back `world_state`, `status`, and `execution_history`.

In [2]:
from langgoap.integrations import GoapSubgraph

sub = GoapSubgraph(actions=quickstart_actions(), goal=goal)
compiled = sub.compile()
result = compiled.invoke({"world_state": {"topic": "GOAP"}, "goal": goal})

print("status:    ", result["status"])
print("executed:  ", [r.action_name for r in result["execution_history"]])
print("brief:     ", result["world_state"]["brief"])
print("draft:     ", result["world_state"]["draft"])
print("url:       ", result["world_state"]["url"])
print("published: ", result["world_state"]["published"])

status:     goal_achieved
executed:   ['research', 'write', 'publish']
brief:      Brief on GOAP
draft:      Article from: Brief on GOAP
url:        https://blog/Article fr
published:  True


## 3. Embed in a parent `StateGraph`

This is where Layer C earns its keep. `add_goap_subgraph` attaches the planner as a single node inside any LangGraph app. The parent's state schema declares `world_state` (input to the planner) and `plan_result` (output back from the planner); no other GOAP keys are exposed.

In [3]:
from typing import Any

from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict

from langgoap.integrations import add_goap_subgraph


class ParentState(TypedDict, total=False):
    user_id: str
    world_state: dict[str, Any]
    plan_result: dict[str, Any]
    final_message: str


def entry(state: ParentState) -> dict[str, Any]:
    return {"world_state": {"topic": "GOAP for LangGraph"}}


def finish(state: ParentState) -> dict[str, Any]:
    url = state.get("plan_result", {}).get("world_state", {}).get("url", "<none>")
    return {"final_message": f"hi {state.get('user_id', 'anon')} - published at {url}"}


parent_builder: StateGraph = StateGraph(ParentState)
parent_builder.add_node("entry", entry)
add_goap_subgraph(
    parent_builder,
    name="goap_planner",
    actions=quickstart_actions(),
    goal=goal,
)
parent_builder.add_node("finish", finish)
parent_builder.add_edge(START, "entry")
parent_builder.add_edge("entry", "goap_planner")
parent_builder.add_edge("goap_planner", "finish")
parent_builder.add_edge("finish", END)

parent = parent_builder.compile()
parent_result = parent.invoke({"user_id": "u42"})

print("final_message:", parent_result["final_message"])
print("parent keys:  ", sorted(parent_result.keys()))

final_message: hi u42 - published at https://blog/Article fr
parent keys:   ['final_message', 'plan_result', 'user_id', 'world_state']


### Sealed internals

Notice what's *not* at the parent's top level: `plan`, `current_step`, `execution_history`, `blacklisted_actions`, `action_failure_counts`. The full planning trace is still reachable for debugging — it lives one level deeper under `plan_result`.

In [4]:
goap_private = {
    "plan",
    "current_step",
    "execution_history",
    "blacklisted_actions",
    "action_failure_counts",
}

print("leaked to parent top-level:", sorted(goap_private & set(parent_result.keys())))
print("reachable via plan_result: ", sorted(goap_private & set(parent_result["plan_result"].keys())))
print("plan_result.status:        ", parent_result["plan_result"]["status"])

leaked to parent top-level: []
reachable via plan_result:  ['current_step', 'execution_history', 'plan']
plan_result.status:         goal_achieved


## 4. Custom input / output keys

The default `world_state` / `plan_result` slot names rarely conflict, but when they do, override them. The sub-graph reads the initial world state from `input_key` and writes its final state into `output_key`.

In [5]:
class CustomState(TypedDict, total=False):
    biz_input: dict[str, Any]
    biz_output: dict[str, Any]


def custom_entry(state: CustomState) -> dict[str, Any]:
    return {"biz_input": {"topic": "GOAP"}}


custom_builder: StateGraph = StateGraph(CustomState)
custom_builder.add_node("entry", custom_entry)
add_goap_subgraph(
    custom_builder,
    name="goap",
    actions=quickstart_actions(),
    goal=goal,
    input_key="biz_input",
    output_key="biz_output",
)
custom_builder.add_edge(START, "entry")
custom_builder.add_edge("entry", "goap")
custom_builder.add_edge("goap", END)

custom_result = custom_builder.compile().invoke({})
print("biz_output.status:                ", custom_result["biz_output"]["status"])
print("biz_output.world_state.published: ", custom_result["biz_output"]["world_state"]["published"])

biz_output.status:                 goal_achieved
biz_output.world_state.published:  True


## Next steps

- Mix layers: build the `actions` list with `goapify_tool` when you already have LangChain `@tool`-decorated functions, then pass it to `GoapSubgraph`.
- Use a natural-language goal: resolve the NL goal into a `GoalSpec` *before* constructing the sub-graph (see [`nl_goal_interpreter.ipynb`](nl_goal_interpreter.ipynb)).
- Inspect the planner's choices via `plan_result["plan"]` and `plan_result["execution_history"]` — both are available inside `finish` for telemetry or human-in-the-loop gates.